# vLLM 多轮对话带历史记忆 — 极简入门 Demo

**主题**：在本地用 [vLLM](https://github.com/vllm-project/vllm) 提供 OpenAI 兼容接口，实现“多轮对话 + 历史记忆”最小闭环；仅做入门与轻量实战，不涉及源码改写或高性能优化。

**你将学到**：如何启动 vLLM 服务、如何在 `messages` 中维护会话历史、如何做最小历史裁剪避免上下文无限膨胀。

**开源说明**：本 Notebook 的讲解与示例代码为原创内容，可用于学习、教学和发布到 [Gitee](https://gitee.com)；请同时遵守模型权重与第三方依赖库各自许可证。

---

## 适用显卡与软件环境（新手按表选型）

| 项目 | 说明 |
|------|------|
| **显卡** | **NVIDIA GPU**（推荐 Turing / Ampere / Ada 等）；本 Demo 按单卡 CUDA 场景编写。 |
| **显存** | 默认模型 `Qwen/Qwen2.5-0.5B-Instruct`：建议 **>= 6GB**，更稳妥为 **>= 8GB**；若改 7B 模型，通常建议 >= 16GB。 |
| **驱动 / CUDA** | 驱动需与 PyTorch / vLLM 预编译包匹配；常见组合是 CUDA 11.8 / 12.x 对应官方 PyTorch wheel。 |
| **Python** | 代码语法按 **Python 3.8+** 编写；注意不同 `vllm` 版本可能要求更高 Python 小版本，以安装报错为准。 |
| **系统** | Linux x86_64 最省心；Windows 建议在 WSL2 或单独终端运行 `vllm serve`。 |

---

In [1]:
## 完整依赖安装命令（终端执行，建议先建虚拟环境）

```bash
# 0) 创建并激活虚拟环境
python3 -m venv .venv
source .venv/bin/activate
# Windows: .venv\Scripts\activate

# 1) 升级 pip
python -m pip install --upgrade pip

# 2) 安装与 CUDA 匹配的 PyTorch（示例：CUDA 12.4）
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
# 若你是 CUDA 11.8，可改为：
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# 3) 安装 vLLM + OpenAI SDK + httpx（常见依赖）
pip install "vllm>=0.6.0" "openai>=1.0.0" "httpx>=0.24.0"

# 如果需要一键安装带vllm和pytorch的cuda环境，可以运行docker run --gpus all --net=host --pid=host --ipc=host --privileged --env "HF_TOKEN=$HF_TOKEN" vllm/vllm-openai:latest --entrypoint=/bin/bash
# 4) 运行本 Notebook 需要 Jupyter（任选其一）
pip install jupyter
```

> 说明：若 `vllm` 安装失败，请按终端提示检查 Python 小版本与 CUDA/torch 组合是否匹配。

---

SyntaxError: invalid character '：' (U+FF1A) (216952765.py, line 25)

## 简单使用说明（本地 / Gitee）

1. 克隆或下载仓库，进入目录后按上一节先完成依赖安装并激活虚拟环境。
2. 在本jupyter notebook所在路径下，启动 Jupyter：`jupyter-lab --ip=0.0.0.0 --port=8888 --no-browser --allow-root`。
3. **方式 A**：在系统终端先启动 `vllm serve ...`，Notebook 只跑客户端单元。
4. **方式 B**：按顺序执行本 Notebook：环境自检 → 启动服务 → 多轮对话 → 停止服务。
5. 首次运行会下载模型权重，需网络可达或已配置镜像缓存。

---

In [2]:
# 可选：在 Notebook 当前内核中安装依赖（若终端已安装可跳过）
# 这一格风格与同目录其他 Demo 保持一致

import subprocess  # 用子进程调用 pip
import sys  # 获取当前内核解释器路径

# 要安装的包列表（固定最小版本）
packages = ["vllm>=0.6.0", "openai>=1.0.0", "httpx>=0.24.0", "requests>=2.31.0"]

# 对当前解释器执行 pip install，避免装到错误环境
cmd = [sys.executable, "-m", "pip", "install", "--upgrade", "pip"] + packages
subprocess.run(cmd, check=True)

print("Notebook 内依赖安装完成；若 torch/cuda 不匹配，请按上文单独安装 PyTorch。")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/432.3 MB ? eta -:--:--

   ━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/432.3 MB 256.6 MB/s eta 0:00:02

   ━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.9/432.3 MB 274.4 MB/s eta 0:00:02

   ━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━ 162.5/432.3 MB 277.8 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━ 217.8/432.3 MB 276.5 MB/s eta 0:00:01

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 3/4 [vllm]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 3/4 [vllm]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 3/4 [vllm]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [vllm]


Notebook 内依赖安装完成；若 torch/cuda 不匹配，请按上文单独安装 PyTorch。


# 环境快速自检：Python / torch / CUDA 可见性

import sys

print("Python:", sys.version)

try:
    import torch
except ImportError:
    print("未检测到 torch：请先按上文安装带 CUDA 的 PyTorch。")
else:
    print("torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU0:", torch.cuda.get_device_name(0))
        total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print("GPU0 显存(GB, 约):", round(total_mem_gb, 2))

## 启动 vLLM OpenAI 兼容服务（Notebook 子进程）

此单元会：
1. 启动 vLLM 服务进程
2. 轮询 `/v1/models` 直到服务可用
3. 启动失败时打印日志尾部，帮助定位问题

若你已在终端手动运行 `vllm serve ...`，可跳过下一格启动代码，只执行后续客户端对话单元。

---

In [3]:
# 在 Notebook 内启动 vLLM（含参数定义与健康检查）

import os
import json
import time
import shutil
import subprocess
from pathlib import Path
import urllib.error
import urllib.request

# ----- 可按需修改的参数 -----
HOST = "127.0.0.1"
PORT = 8000
MODEL_NAME = os.environ.get("DEMO_MODEL_NAME", "Qwen/Qwen2.5-0.5B-Instruct")
MAX_MODEL_LEN = 2048
GPU_MEMORY_UTILIZATION = 0.90
DTYPE = "half"  # 入门场景常用 half，显存更友好
LOG_PATH = Path("./vllm_server_demo.log")
# -------------------------

# 如果之前已启动进程，先停止，避免端口冲突
if "vllm_process" in globals() and vllm_process is not None and vllm_process.poll() is None:
    print("检测到旧的 vLLM 进程，先停止后重启...")
    vllm_process.terminate()
    try:
        vllm_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        vllm_process.kill()
        vllm_process.wait(timeout=5)

# 打开日志文件，保存服务输出
log_file = open(LOG_PATH, "w", encoding="utf-8")

# 优先使用 vllm CLI；若 PATH 中无 vllm，则回退 python -m
vllm_bin = shutil.which("vllm")
if vllm_bin:
    cmd = [
        vllm_bin,
        "serve",
        MODEL_NAME,
        "--host",
        HOST,
        "--port",
        str(PORT),
        "--dtype",
        DTYPE,
        "--max-model-len",
        str(MAX_MODEL_LEN),
        "--gpu-memory-utilization",
        str(GPU_MEMORY_UTILIZATION),
    ]
else:
    cmd = [
        os.environ.get("PYTHON", "python"),
        "-m",
        "vllm.entrypoints.openai.api_server",
        "--host",
        HOST,
        "--port",
        str(PORT),
        "--model",
        MODEL_NAME,
        "--dtype",
        DTYPE,
        "--max-model-len",
        str(MAX_MODEL_LEN),
        "--gpu-memory-utilization",
        str(GPU_MEMORY_UTILIZATION),
    ]

print("启动命令:", " ".join(cmd))

vllm_process = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=log_file,
    start_new_session=True,
)

health_url = f"http://{HOST}:{PORT}/v1/models"
deadline = time.time() + 600  # 首次拉模型可能较慢，最多等 10 分钟

while time.time() < deadline:
    if vllm_process.poll() is not None:
        log_file.flush()
        with open(LOG_PATH, "r", encoding="utf-8", errors="ignore") as f:
            logs = f.read()
        raise RuntimeError(
            "vLLM 服务启动失败。日志尾部如下:\n"
            + "\n".join(logs.splitlines()[-100:])
        )
    try:
        with urllib.request.urlopen(health_url, timeout=2) as resp:
            body = resp.read().decode("utf-8")
        data = json.loads(body)
        print("服务已就绪，/v1/models 顶层键:", list(data.keys()))
        print("OpenAI 兼容 Base URL:", f"http://{HOST}:{PORT}/v1")
        print("日志文件:", LOG_PATH.resolve())
        break
    except (urllib.error.URLError, urllib.error.HTTPError, TimeoutError, json.JSONDecodeError):
        time.sleep(2)
else:
    raise TimeoutError(f"等待 vLLM 服务超时，请查看日志: {LOG_PATH.resolve()}")

启动命令: /usr/local/bin/vllm serve Qwen/Qwen2.5-0.5B-Instruct --host 127.0.0.1 --port 8000 --dtype half --max-model-len 2048 --gpu-memory-utilization 0.9


服务已就绪，/v1/models 顶层键: ['object', 'data']
OpenAI 兼容 Base URL: http://127.0.0.1:8000/v1
日志文件: /workspace/community_materials_nb/vllm_server_demo.log


## 多轮对话 + 历史记忆（核心）

要点：
- 用 `messages` 列表保存历史对话；
- 每次新提问前追加一条 `user` 消息；
- 每次模型回复后追加一条 `assistant` 消息；
- 请求前做历史裁剪，避免上下文无限增长。

---

In [4]:
# 导入 OpenAI 客户端（用于调用 vLLM 的 OpenAI 兼容接口）
from openai import OpenAI

# 若你跳过了 Notebook 启动单元，可在这里直接指定服务地址
HOST = globals().get("HOST", "127.0.0.1")
PORT = globals().get("PORT", 8000)

# 创建客户端，base_url 指向本机 vLLM 服务
client = OpenAI(
    api_key="EMPTY",  # vLLM 本地服务默认不校验真实 key，这里给占位字符串即可
    base_url=f"http://{HOST}:{PORT}/v1",
)

# 读取当前服务可用模型列表
model_list = client.models.list().data

# 选择第一个可用模型作为推理模型 ID
SERVING_MODEL = model_list[0].id

# 打印模型 ID，方便确认
print("SERVING_MODEL =", SERVING_MODEL)

# 初始化会话历史（system 角色用于设定助手风格）
conversation_history = [
    {
        "role": "system",
        "content": "你是一个简洁、可靠的中文技术助手。",
    }
]

# 定义历史裁剪函数，避免历史无限增长
def trim_history(history, max_turns=6):
    # 取出 system 消息（通常只保留第一条）
    system_msgs = [m for m in history if m["role"] == "system"][:1]

    # 取出非 system 的对话消息（user/assistant）
    dialog_msgs = [m for m in history if m["role"] != "system"]

    # 仅保留最近 max_turns 轮（每轮约 2 条消息）
    keep_count = max_turns * 2
    dialog_msgs = dialog_msgs[-keep_count:]

    # 返回裁剪后的历史
    return system_msgs + dialog_msgs

# 定义单轮对话函数：输入用户问题，返回模型回答
def chat_once(user_text, temperature=0.7, max_tokens=256):
    # 将用户输入写入历史（这是“带记忆”的关键之一）
    conversation_history.append({"role": "user", "content": user_text})

    # 在请求前先裁剪历史，控制上下文长度
    trimmed = trim_history(conversation_history, max_turns=6)

    # 发起 chat.completions 请求
    response = client.chat.completions.create(
        model=SERVING_MODEL,
        messages=trimmed,
        temperature=temperature,
        max_tokens=max_tokens,
    )

    # 读取模型回答文本
    assistant_text = response.choices[0].message.content

    # 将模型回答写回历史（这是“带记忆”的关键之二）
    conversation_history.append({"role": "assistant", "content": assistant_text})

    # 再次裁剪并回写，避免历史持续膨胀
    new_history = trim_history(conversation_history, max_turns=6)
    conversation_history.clear()
    conversation_history.extend(new_history)

    # 返回当前回答
    return assistant_text

SERVING_MODEL = Qwen/Qwen2.5-0.5B-Instruct


In [5]:
# 第 1 轮：先告诉模型一个“需要记住”的偏好
q1 = "从现在开始，请记住我喜欢低糖咖啡。先回复：已记住。"
a1 = chat_once(q1)
print("[用户]", q1)
print("[助手]", a1)
print("-" * 60)

# 第 2 轮：问一个新问题
q2 = "请给我推荐两种适合早餐的饮品。"
a2 = chat_once(q2)
print("[用户]", q2)
print("[助手]", a2)
print("-" * 60)

# 第 3 轮：检查模型是否利用历史记忆
q3 = "结合我之前的偏好，再给我一个饮品建议，并说明原因。"
a3 = chat_once(q3)
print("[用户]", q3)
print("[助手]", a3)
print("-" * 60)

# 输出当前历史条数，验证多轮消息确实被保存
print("当前历史消息条数:", len(conversation_history))
for i, msg in enumerate(conversation_history):
    print(f"{i:02d}. {msg['role']}: {msg['content'][:80]}")

[用户] 从现在开始，请记住我喜欢低糖咖啡。先回复：已记住。
[助手] 已记住。
------------------------------------------------------------
[用户] 请给我推荐两种适合早餐的饮品。
[助手] 当然可以，以下是两种适合早餐的饮品：

1. 绿茶：绿茶是一种健康的饮品，含有抗氧化剂和多种维生素。它能帮助提神醒脑，提高注意力。

2. 水果奶昔：将水果和牛奶混合制成美味的饮料，既健康又可口。可以选择一些低糖或无糖的选择，以减少热量摄入。
------------------------------------------------------------


[用户] 结合我之前的偏好，再给我一个饮品建议，并说明原因。
[助手] 好的，根据您的喜好，以下是一些建议：

1. **热巧克力**：
   - 原因：热巧克力是温暖且舒适的饮品选择，能够提升心情并提供能量。
   - 选择理由：热巧克力通常含有丰富的脂肪和碳水化合物，但其热量较低，更适合需要快速能量补充的人群。同时，它也可以搭配其他口味的饮品一起享用，增加口感层次。

2. **豆浆**：
   - 原因：豆浆是一种营养丰富且易于消化的饮品，富含蛋白质、铁质等营养素，有助于增强体力和免疫力。
   - 选择理由：豆浆不含过多的糖分和油脂，而且它的热量相对较低，适合追求健康饮食习惯的人士。此外，豆浆还可以与其他食物如坚果、酸奶等搭配食用，增添风味和营养价值。

希望这些建议对您有所帮助！如果您有更多具体需求或者偏好，欢迎随时告诉我。
------------------------------------------------------------
当前历史消息条数: 7
00. system: 你是一个简洁、可靠的中文技术助手。
01. user: 从现在开始，请记住我喜欢低糖咖啡。先回复：已记住。
02. assistant: 已记住。
03. user: 请给我推荐两种适合早餐的饮品。
04. assistant: 当然可以，以下是两种适合早餐的饮品：

1. 绿茶：绿茶是一种健康的饮品，含有抗氧化剂和多种维生素。它能帮助提神醒脑，提高注意力。

2. 水果奶昔：将水果和牛
05. user: 结合我之前的偏好，再给我一个饮品建议，并说明原因。
06. assistant: 好的，根据您的喜好，以下是一些建议：

1. **热巧克力**：
   - 原因：热巧克力是温暖且舒适的饮品选择，能够提升心情并提供能量。
   - 选择理由：


## 常见报错与解决方案（社区速查）

### 1）`RuntimeError: CUDA out of memory`

**原因**：模型规模、上下文长度或并行占用超过当前显存。  
**处理**：
- 将 `MAX_MODEL_LEN` 从 `2048` 调低到 `1024` 或更低；
- 将 `GPU_MEMORY_UTILIZATION` 从 `0.90` 调低到 `0.80` 左右；
- 优先使用本 Demo 默认的小模型 `Qwen/Qwen2.5-0.5B-Instruct`；
- 用 `nvidia-smi` 清理其它占显存进程。

### 2）`Connection refused` / `All connection attempts failed` / 请求超时

**原因**：vLLM 服务尚未就绪、端口不一致或端口被占用。  
**处理**：
- 确保启动单元已经打印“服务已就绪”；
- 确保客户端 `base_url` 与 `HOST` / `PORT` 完全一致；
- 若端口冲突，将 `PORT=8000` 改为 `8001` 等空闲端口；
- 查看 `vllm_server_demo.log` 末尾日志定位根因。

---

**执行建议**：完整流程推荐按以下顺序运行：依赖安装（终端）→ 可选 Notebook 安装 → 环境自检 → 启动服务 → 多轮对话 → 停止服务。

In [6]:
# 结束实验后执行本单元，释放显存

import subprocess

if "vllm_process" in globals() and vllm_process is not None and vllm_process.poll() is None:
    print("正在停止 vLLM 服务进程...")
    vllm_process.terminate()
    try:
        vllm_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        print("进程未及时退出，执行强制结束...")
        vllm_process.kill()
        vllm_process.wait(timeout=5)
    print("vLLM 服务已停止。")
else:
    print("未检测到正在运行的 Notebook 子进程（若你在终端启动，请在终端 Ctrl+C 停止）。")

if "log_file" in globals() and log_file and not log_file.closed:
    log_file.close()
    print("日志文件句柄已关闭。")

正在停止 vLLM 服务进程...


vLLM 服务已停止。
日志文件句柄已关闭。
